# NYC EMS Silver Data Cleaning and Transformation

## Purpose

This notebook transforms the raw Bronze EMS incident table into a
validated, analysis-ready Silver Delta table.

## Responsibilities

- Read the Bronze Delta table
- Validate incident identifier uniqueness
- Trim text and convert empty strings to null
- Convert date-time and numeric columns
- Standardise indicator values
- Create valid response-time measures
- Derive date and operational analysis fields
- Preserve invalid records using data-quality flags
- Write a partitioned Silver Delta table

## Input

- `bronze_ems_incidents`

## Planned output

- `silver_ems_incidents`
- `silver_ems_data_quality_audit`

In [2]:
# Import PySpark component
from pyspark.sql import functions as F
from pyspark.sql.types import(
    StringType,
    IntegerType,
    DoubleType,
    TimestampType,
    DataType,
    BooleanType
)

StatementMeta(, 1c1a96c0-948f-4f2f-879d-8aeb5e83b856, 4, Finished, Available, Finished, False)

In [3]:
# Table names
bronze_table_name = "bronze_ems_incidents"
silver_table_name = "silver_ems_incidents"
silver_audit_table_name = "silver_ems_data_quality_audit"

# Read Bronze table
df_bronze = spark.table(bronze_table_name)

print(f"Input table: {bronze_table_name}")
print(f"Bronze columns: {len(df_bronze.columns)}")

df_bronze.printSchema()

StatementMeta(, 1c1a96c0-948f-4f2f-879d-8aeb5e83b856, 5, Finished, Available, Finished, False)

Input table: bronze_ems_incidents
Bronze columns: 34
root
 |-- incident_id: string (nullable = true)
 |-- incident_datetime: string (nullable = true)
 |-- initial_call_type: string (nullable = true)
 |-- initial_severity_level_code: string (nullable = true)
 |-- final_call_type: string (nullable = true)
 |-- final_severity_level_code: string (nullable = true)
 |-- first_assignment_datetime: string (nullable = true)
 |-- valid_dispatch_rspns_time_indc: string (nullable = true)
 |-- dispatch_response_seconds_qy: string (nullable = true)
 |-- first_activation_datetime: string (nullable = true)
 |-- first_on_scene_datetime: string (nullable = true)
 |-- valid_incident_rspns_time_indc: string (nullable = true)
 |-- incident_response_seconds_qy: string (nullable = true)
 |-- incident_travel_tm_seconds_qy: string (nullable = true)
 |-- first_to_hosp_datetime: string (nullable = true)
 |-- first_hosp_arrival_datetime: string (nullable = true)
 |-- incident_close_datetime: string (nullable = tr

In [4]:
# Check the original schema
display(
    df_bronze.select(
        "incident_id",
        "incident_datetime",
        "initial_call_type",
        "initial_severity_level_code",
        "dispatch_response_seconds_qy",
        "incident_response_seconds_qy",
        "borough",
        "held_indicator",
        "_source_year"
    ).limit(20)
)

StatementMeta(, 1c1a96c0-948f-4f2f-879d-8aeb5e83b856, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 71e6624f-b34e-4bad-9db9-86437d833a9b)

## Primary Key Validation

`incident_id` is described by the source as the unique identifier for
an EMS incident. The Silver transformation verifies that it is both
complete and unique before deduplication.

In [5]:
# Primary key validation
incident_id_statistics = (
    df_bronze
    .agg(
        F.count("*").alias("total_rows"),
        F.count("incident_id").alias("non_null_incident_ids"),
        F.countDistinct("incident_id").alias("distinct_incident_ids")
    )
    .collect()[0]
)

bronze_total_rows = incident_id_statistics["total_rows"]
non_null_incident_ids = incident_id_statistics["non_null_incident_ids"]
distinct_incident_ids = incident_id_statistics["distinct_incident_ids"]

missing_incident_ids = bronze_total_rows - non_null_incident_ids
duplicate_incident_rows = (
    non_null_incident_ids - distinct_incident_ids
)

print(f"Total rows:              {bronze_total_rows:,}")
print(f"Non-null incident IDs:   {non_null_incident_ids:,}")
print(f"Distinct incident IDs:   {distinct_incident_ids:,}")
print(f"Missing incident IDs:    {missing_incident_ids:,}")
print(f"Duplicate incident rows: {duplicate_incident_rows:,}")

StatementMeta(, 1c1a96c0-948f-4f2f-879d-8aeb5e83b856, 7, Finished, Available, Finished, False)

Total rows:              10,881,496
Non-null incident IDs:   10,881,496
Distinct incident IDs:   10,881,496
Missing incident IDs:    0
Duplicate incident rows: 0


In [6]:
# Add a uniqueness assertion
assert bronze_total_rows == 10881496, (
    f"Expected 10,881,496 rows, found {bronze_total_rows}"
)

assert missing_incident_ids == 0, (
    f"Found {missing_incident_ids} missing incident IDs"
)

assert duplicate_incident_rows == 0, (
    f"Found {duplicate_incident_rows} duplicate incident rows"
)

print("Incident ID validation passed.")

StatementMeta(, 1c1a96c0-948f-4f2f-879d-8aeb5e83b856, 8, Finished, Available, Finished, False)

Incident ID validation passed.


In [7]:
# Check the value of the Indicator field
indicator_columns = [
    "valid_dispatch_rspns_time_indc",
    "valid_incident_rspns_time_indc",
    "held_indicator",
    "reopen_indicator",
    "special_event_indicator",
    "standby_indicator",
    "transfer_indicator"
]

for column_name in indicator_columns:
    print(f"Values in {column_name}:")

    (
        df_bronze
        .groupBy(column_name)
        .count()
        .orderBy(F.col("count").desc())
        .show(truncate=False)
    )

StatementMeta(, 1c1a96c0-948f-4f2f-879d-8aeb5e83b856, 9, Finished, Available, Finished, False)

Values in valid_dispatch_rspns_time_indc:
+------------------------------+--------+
|valid_dispatch_rspns_time_indc|count   |
+------------------------------+--------+
|Y                             |10772516|
|N                             |108980  |
+------------------------------+--------+

Values in valid_incident_rspns_time_indc:
+------------------------------+--------+
|valid_incident_rspns_time_indc|count   |
+------------------------------+--------+
|Y                             |10425070|
|N                             |456426  |
+------------------------------+--------+

Values in held_indicator:
+--------------+-------+
|held_indicator|count  |
+--------------+-------+
|N             |9687505|
|Y             |1193991|
+--------------+-------+

Values in reopen_indicator:
+----------------+--------+
|reopen_indicator|count   |
+----------------+--------+
|N               |10707670|
|Y               |173826  |
+----------------+--------+

Values in special_event_indicator:
+

In [8]:
# Check for date and numerical conversion risks
datetime_columns = [
    "incident_datetime",
    "first_assignment_datetime",
    "first_activation_datetime",
    "first_on_scene_datetime",
    "first_to_hosp_datetime",
    "first_hosp_arrival_datetime",
    "incident_close_datetime"
]

numeric_columns = [
    "initial_severity_level_code",
    "final_severity_level_code",
    "dispatch_response_seconds_qy",
    "incident_response_seconds_qy",
    "incident_travel_tm_seconds_qy"
]

datetime_validation_expressions = [
    F.sum(
        F.when(
            F.col(column_name).isNotNull() &
            F.to_timestamp(F.col(column_name)).isNull(),
            1
        ).otherwise(0)
    ).alias(f"{column_name}_invalid")
    for column_name in datetime_columns
]

numeric_validation_expressions = [
    F.sum(
        F.when(
            F.col(column_name).isNotNull() &
            F.col(column_name).cast("double").isNull(),
            1
        ).otherwise(0)
    ).alias(f"{column_name}_invalid")
    for column_name in numeric_columns
]

conversion_validation = (
    df_bronze
    .agg(
        *datetime_validation_expressions,
        *numeric_validation_expressions
    )
)

display(conversion_validation)

StatementMeta(, 1c1a96c0-948f-4f2f-879d-8aeb5e83b856, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5b966411-d43c-4b4a-ade2-22e12ca25aca)

## Clean Text Values

Text values are trimmed and empty strings are converted to null.
Indicator fields are standardised to uppercase while preserving their
original Y/N representation.

In [9]:
df_silver = df_bronze

# Trim every string column and convert empty strings to null
for field in df_silver.schema.fields:
    if isinstance(field.dataType, StringType):
        column_name = field.name

        df_silver = df_silver.withColumn(
            column_name,
            F.when(
                F.trim(F.col(column_name)) == "",
                F.lit(None)
            ).otherwise(
                F.trim(F.col(column_name))
            )
        )

indicator_columns = [
    "valid_dispatch_rspns_time_indc",
    "valid_incident_rspns_time_indc",
    "held_indicator",
    "reopen_indicator",
    "special_event_indicator",
    "standby_indicator",
    "transfer_indicator"
]

# Standardise indicator values
for column_name in indicator_columns:
    df_silver = df_silver.withColumn(
        column_name,
        F.upper(F.col(column_name))
    )

print("Text cleaning rules applied.")

StatementMeta(, 1c1a96c0-948f-4f2f-879d-8aeb5e83b856, 11, Finished, Available, Finished, False)

Text cleaning rules applied.


## Convert Date-Time Columns

All EMS timestamps are treated as NYC local operational timestamps.
No conversion to New Zealand time or UTC is applied because the source
does not provide a timezone offset.

In [10]:
datetime_columns = [
    "incident_datetime",
    "first_assignment_datetime",
    "first_activation_datetime",
    "first_on_scene_datetime",
    "first_to_hosp_datetime",
    "first_hosp_arrival_datetime",
    "incident_close_datetime"
]

for column_name in datetime_columns:
    df_silver = df_silver.withColumn(
        column_name,
        F.to_timestamp(F.col(column_name))
    )

print("Date-time conversions applied.")

StatementMeta(, 1c1a96c0-948f-4f2f-879d-8aeb5e83b856, 12, Finished, Available, Finished, False)

Date-time conversions applied.


In [11]:
# Convert numeric columns
integer_columns = [
    "initial_severity_level_code",
    "final_severity_level_code"
]

response_time_columns = [
    "dispatch_response_seconds_qy",
    "incident_response_seconds_qy",
    "incident_travel_tm_seconds_qy"
]

for column_name in integer_columns:
    df_silver = df_silver.withColumn(
        column_name,
        F.col(column_name).cast("int")
    )

for column_name in response_time_columns:
    df_silver = df_silver.withColumn(
        column_name,
        F.col(column_name).cast("double")
    )

print("Numeric conversions applied.")

StatementMeta(, 1c1a96c0-948f-4f2f-879d-8aeb5e83b856, 13, Finished, Available, Finished, False)

Numeric conversions applied.


In [12]:
# Create boolean columns
def yn_to_boolean(column_name):
    return (
        F.when(F.col(column_name) == "Y", F.lit(True))
        .when(F.col(column_name) == "N", F.lit(False))
        .otherwise(F.lit(None).cast("boolean"))
    )

df_silver = (
    df_silver
    .withColumn(
        "is_valid_dispatch_response",
        yn_to_boolean("valid_dispatch_rspns_time_indc")
    )
    .withColumn(
        "is_valid_incident_response",
        yn_to_boolean("valid_incident_rspns_time_indc")
    )
    .withColumn(
        "is_held",
        yn_to_boolean("held_indicator")
    )
    .withColumn(
        "is_reopened",
        yn_to_boolean("reopen_indicator")
    )
    .withColumn(
        "is_special_event",
        yn_to_boolean("special_event_indicator")
    )
    .withColumn(
        "is_standby",
        yn_to_boolean("standby_indicator")
    )
    .withColumn(
        "is_transfer",
        yn_to_boolean("transfer_indicator")
    )
)

print("Boolean indicator columns created.")

StatementMeta(, 1c1a96c0-948f-4f2f-879d-8aeb5e83b856, 14, Finished, Available, Finished, False)

Boolean indicator columns created.


In [13]:
# Create an Effective Response Time field
df_silver = (
    df_silver
    .withColumn(
        "valid_dispatch_response_seconds",
        F.when(
            (F.col("is_valid_dispatch_response") == True) &
            (F.col("dispatch_response_seconds_qy") >= 0),
            F.col("dispatch_response_seconds_qy")
        )
    )
    .withColumn(
        "valid_incident_response_seconds",
        F.when(
            (F.col("is_valid_incident_response") == True) &
            (F.col("incident_response_seconds_qy") >= 0),
            F.col("incident_response_seconds_qy")
        )
    )
    .withColumn(
        "valid_travel_time_seconds",
        F.when(
            F.col("incident_travel_tm_seconds_qy") >= 0,
            F.col("incident_travel_tm_seconds_qy")
        )
    )
)

print("Valid response-time measures created.")

StatementMeta(, 1c1a96c0-948f-4f2f-879d-8aeb5e83b856, 15, Finished, Available, Finished, False)

Valid response-time measures created.


In [14]:
# Create Date Analysis Fields
df_silver = (
    df_silver
    .withColumn(
        "incident_date",
        F.to_date("incident_datetime")
    )
    .withColumn(
        "incident_year",
        F.year("incident_datetime")
    )
    .withColumn(
        "incident_quarter",
        F.quarter("incident_datetime")
    )
    .withColumn(
        "incident_month",
        F.month("incident_datetime")
    )
    .withColumn(
        "incident_month_name",
        F.date_format("incident_datetime", "MMMM")
    )
    .withColumn(
        "incident_year_month",
        F.date_format("incident_datetime", "yyyy-MM")
    )
    .withColumn(
        "incident_day_of_week_number",
        F.dayofmonth("incident_datetime")
    )
    .withColumn(
        "incident_day_of_week_name",
        F.date_format("incident_datetime", "EEEE")
    )
    .withColumn(
        "incident_hour",
        F.hour("incident_datetime")
    )
    .withColumn(
        "is_weekend",
        F.dayofweek("incident_datetime").isin([1, 7])
    )
)

print("Date analysis columns created.")

StatementMeta(, 1c1a96c0-948f-4f2f-879d-8aeb5e83b856, 16, Finished, Available, Finished, False)

Date analysis columns created.


In [15]:
# Create Time of Day
df_silver = df_silver.withColumn(
    "time_of_day",
    F.when(
        F.col("incident_hour").between(0, 5),
        "Overnight"
    )
    .when(
        F.col("incident_hour").between(6, 11),
        "Morning"
    )
    .when(
        F.col("incident_hour").between(12, 17),
        "Afternoon"
    )
    .otherwise("Evening")
)

print("Time-of-day category created.")

StatementMeta(, 1c1a96c0-948f-4f2f-879d-8aeb5e83b856, 17, Finished, Available, Finished, False)

Time-of-day category created.


In [16]:
# Create a field with minute-level granularity
df_silver = (
    df_silver
    .withColumn(
        "dispatch_response_minutes",
        F.round(
            F.col("valid_dispatch_response_seconds") / 60.0,
            2
        )
    )
    .withColumn(
        "incident_response_minutes",
        F.round(
            F.col("valid_incident_response_seconds") / 60.0,
            2
        )
    )
    .withColumn(
        "travel_time_minutes",
        F.round(
            F.col("valid_travel_time_seconds") / 60.0,
            2
        )
    )
)

print("Response-time minute columns created.")

StatementMeta(, 1c1a96c0-948f-4f2f-879d-8aeb5e83b856, 18, Finished, Available, Finished, False)

Response-time minute columns created.


In [17]:
# Create fields for Call Type and Severity changes
df_silver = (
    df_silver
    .withColumn(
        "call_type_changed",
        F.when(
            F.col("initial_call_type").isNull() |
            F.col("final_call_type").isNull(),
            F.lit(None).cast("boolean")
        ).otherwise(
            F.col("initial_call_type") !=
            F.col("final_call_type")
        )
    )
    .withColumn(
        "severity_changed",
        F.when(
            F.col("initial_severity_level_code").isNull() |
            F.col("final_severity_level_code").isNull(),
            F.lit(None).cast("boolean")
        ).otherwise(
            F.col("initial_severity_level_code") !=
            F.col("final_severity_level_code")
        )
    )
)

print("Call-type and severity-change indicators created.")

StatementMeta(, 1c1a96c0-948f-4f2f-879d-8aeb5e83b856, 19, Finished, Available, Finished, False)

Call-type and severity-change indicators created.


In [18]:
# Track incident duration and time to hospital arrival
incident_duration_expression = (
    F.col("incident_close_datetime").cast("long") -
    F.col("incident_datetime").cast("long")
)

hospital_travel_expression = (
    F.col("first_hosp_arrival_datetime").cast("long") -
    F.col("first_to_hosp_datetime").cast("long")
)

df_silver = (
    df_silver
    .withColumn(
        "incident_duration_seconds",
        F.when(
            incident_duration_expression >= 0,
            incident_duration_expression .cast("double")
        )
    )
    .withColumn(
        "hospital_travel_seconds",
        F.when(
            hospital_travel_expression >= 0,
            hospital_travel_expression.cast("double")
        )
    )
)

print("Incident-duration fields created.")

StatementMeta(, 1c1a96c0-948f-4f2f-879d-8aeb5e83b856, 20, Finished, Available, Finished, False)

Incident-duration fields created.


In [19]:
# Create record-quality status
df_silver = df_silver.withColumn(
    "record_quality_status",
    F.when(
        F.col("incident_id").isNull(),
        "MISSING_INCIDENT_ID"
    )
    .when(
        F.col("incident_datetime").isNull(),
        "INVALID_INCIDENT_DATETIME"
    )
    .when(
        F.col("incident_year") != F.col("_source_year"),
        "SOURCE_YEAR_MISMATCH"
    )
    .otherwise("VALID")
)

print("Record-quality status created.")

StatementMeta(, 1c1a96c0-948f-4f2f-879d-8aeb5e83b856, 21, Finished, Available, Finished, False)

Record-quality status created.


In [20]:
# Check the resault of Sliver table
print(f"Silver column count: {len(df_silver.columns)}")

df_silver.printSchema()

display(
    df_silver.select(
        "incident_id",
        "incident_datetime",
        "incident_date",
        "incident_year_month",
        "incident_hour",
        "time_of_day",
        "borough",
        "incident_response_seconds_qy",
        "valid_incident_response_seconds",
        "incident_response_minutes",
        "is_held",
        "call_type_changed",
        "record_quality_status"
    ).limit(20)
)

StatementMeta(, 1c1a96c0-948f-4f2f-879d-8aeb5e83b856, 22, Finished, Available, Finished, False)

Silver column count: 63
root
 |-- incident_id: string (nullable = true)
 |-- incident_datetime: timestamp (nullable = true)
 |-- initial_call_type: string (nullable = true)
 |-- initial_severity_level_code: integer (nullable = true)
 |-- final_call_type: string (nullable = true)
 |-- final_severity_level_code: integer (nullable = true)
 |-- first_assignment_datetime: timestamp (nullable = true)
 |-- valid_dispatch_rspns_time_indc: string (nullable = true)
 |-- dispatch_response_seconds_qy: double (nullable = true)
 |-- first_activation_datetime: timestamp (nullable = true)
 |-- first_on_scene_datetime: timestamp (nullable = true)
 |-- valid_incident_rspns_time_indc: string (nullable = true)
 |-- incident_response_seconds_qy: double (nullable = true)
 |-- incident_travel_tm_seconds_qy: double (nullable = true)
 |-- first_to_hosp_datetime: timestamp (nullable = true)
 |-- first_hosp_arrival_datetime: timestamp (nullable = true)
 |-- incident_close_datetime: timestamp (nullable = true)
 |

SynapseWidget(Synapse.DataFrame, 3917e179-b8ff-4494-a26e-757cdbc7d0b1)

In [21]:
# Validate Sliver transformation
silver_validation = (
    df_silver
    .agg(
        F.count("*").alias("total_rows"),

        F.sum(
            F.when(
                F.col("record_quality_status") != "VALID",
                1
            ).otherwise(0)
        ).alias("invalid_records"),

        F.sum(
            F.when(
                F.col("incident_year") != F.col("_source_year"),
                1
            ).otherwise(0)
        ).alias("source_year_mismatches"),

        F.sum(
            F.when(
                F.col("dispatch_response_seconds_qy") < 0,
                1
            ).otherwise(0)
        ).alias("negative_dispatch_times"),

        F.sum(
            F.when(
                F.col("incident_travel_tm_seconds_qy") < 0,
                1
            ).otherwise(0)
        ).alias("negative_travel_times"),
    )
    .collect()[0]
)

for field_name in silver_validation.asDict():
    print(
        f"{field_name}: "
        f"{silver_validation[field_name]:,}"
    )

assert silver_validation["total_rows"] == bronze_total_rows

assert silver_validation["source_year_mismatches"] == 0, (
    "Incident year does not match source partition year."
)

assert silver_validation["invalid_records"] == 0, (
    "One or more records failed Silver quality validation."
)

print("Silver transformation validation passed.")

StatementMeta(, 1c1a96c0-948f-4f2f-879d-8aeb5e83b856, 23, Finished, Available, Finished, False)

total_rows: 10,881,496
invalid_records: 0
source_year_mismatches: 0
negative_dispatch_times: 0
negative_travel_times: 0
Silver transformation validation passed.


In [22]:
# Write df_silver to the Silver Delta table
# Enable write optimizations
spark.conf.set(
    "spark.microsoft.delta.optimizeWrite.enabled",
    "true"
)

spark.conf.set(
    "spark.microsoft.delta.autoCompact.enabled",
    "true"
)

# Write Silver table
(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("incident_year")
    .saveAsTable(silver_table_name)
)

print(f"Successfully created Silver table: {silver_table_name}")

StatementMeta(, 1c1a96c0-948f-4f2f-879d-8aeb5e83b856, 24, Finished, Available, Finished, False)

Successfully created Silver table: silver_ems_incidents


In [23]:
# Read the persisted Silver table
df_silver_saved = spark.table(silver_table_name)

print(f"Successfully loaded: {silver_table_name}")
print(f"Column count: {len(df_silver_saved.columns)}")

df_silver_saved.printSchema()

StatementMeta(, 1c1a96c0-948f-4f2f-879d-8aeb5e83b856, 25, Finished, Available, Finished, False)

Successfully loaded: silver_ems_incidents
Column count: 63
root
 |-- incident_id: string (nullable = true)
 |-- incident_datetime: timestamp (nullable = true)
 |-- initial_call_type: string (nullable = true)
 |-- initial_severity_level_code: integer (nullable = true)
 |-- final_call_type: string (nullable = true)
 |-- final_severity_level_code: integer (nullable = true)
 |-- first_assignment_datetime: timestamp (nullable = true)
 |-- valid_dispatch_rspns_time_indc: string (nullable = true)
 |-- dispatch_response_seconds_qy: double (nullable = true)
 |-- first_activation_datetime: timestamp (nullable = true)
 |-- first_on_scene_datetime: timestamp (nullable = true)
 |-- valid_incident_rspns_time_indc: string (nullable = true)
 |-- incident_response_seconds_qy: double (nullable = true)
 |-- incident_travel_tm_seconds_qy: double (nullable = true)
 |-- first_to_hosp_datetime: timestamp (nullable = true)
 |-- first_hosp_arrival_datetime: timestamp (nullable = true)
 |-- incident_close_datet

In [24]:
# Perform a quality audit on the final Silver table
silver_audit_result = (
    df_silver_saved
    .agg(
        F.count("*").alias("total_rows"),

         F.countDistinct("incident_id").alias(
            "distinct_incident_ids"
        ),

        F.sum(
            F.when(
                F.col("incident_id").isNull(),
                1
            ).otherwise(0)
        ).alias("missing_incident_ids"),

        F.sum(
            F.when(
                F.col("record_quality_status") != "VALID",
                1
            ).otherwise(0)
        ).alias("invalid_records"),

        F.sum(
            F.when(
                F.col("incident_year") != F.col("_source_year"),
                1
            ).otherwise(0)
        ).alias("source_year_mismatches"),

        F.sum(
            F.when(
                F.col("dispatch_response_seconds_qy") < 0,
                1
            ).otherwise(0)
        ).alias("negative_dispatch_times"),

        F.sum(
            F.when(
                F.col("incident_response_seconds_qy") < 0,
                1
            ).otherwise(0)
        ).alias("negative_incident_times"),

        F.sum(
            F.when(
                F.col("incident_travel_tm_seconds_qy") < 0,
                1
            ).otherwise(0)
        ).alias("negative_travel_times"),

        F.min("incident_datetime").alias(
            "minimum_incident_datetime"
        ),

        F.max("incident_datetime").alias(
            "maximum_incident_datetime"
        )
    )
    .first()
)

silver_audit = silver_audit_result.asDict()

silver_audit["duplicate_incident_ids"] = (
    silver_audit["total_rows"]
    - silver_audit["distinct_incident_ids"]
)

silver_audit

StatementMeta(, 1c1a96c0-948f-4f2f-879d-8aeb5e83b856, 26, Finished, Available, Finished, False)

{'total_rows': 10881496,
 'distinct_incident_ids': 10881496,
 'missing_incident_ids': 0,
 'invalid_records': 0,
 'source_year_mismatches': 0,
 'negative_dispatch_times': 0,
 'negative_incident_times': 0,
 'negative_travel_times': 0,
 'minimum_incident_datetime': datetime.datetime(2019, 1, 1, 0, 0, 14),
 'maximum_incident_datetime': datetime.datetime(2025, 12, 31, 23, 59, 35),
 'duplicate_incident_ids': 0}

In [25]:
# Generate audit records
from datetime import datetime
from pyspark.sql import Row

audit_status = (
    "PASS"
    if (
        silver_audit["total_rows"] == bronze_total_rows
        and silver_audit["missing_incident_ids"] == 0
        and silver_audit["duplicate_incident_ids"] == 0
        and silver_audit["invalid_records"] == 0
        and silver_audit["source_year_mismatches"] == 0
        and silver_audit["negative_dispatch_times"] == 0
        and silver_audit["negative_incident_times"] == 0
        and silver_audit["negative_travel_times"] == 0
    )
    else "FAIL"
)

audit_record = Row(
    audit_timestamp = datetime.utcnow(),
    source_table = "bronze_ems_incidents",
    target_table = silver_table_name,
    total_columns = len(df_silver_saved.columns),
    distinct_incident_ids = (
        silver_audit["distinct_incident_ids"]
    ),
    missing_incident_ids = (
        silver_audit["missing_incident_ids"]
    ),
    duplicate_incident_ids = (
        silver_audit["duplicate_incident_ids"]
    ),
    invalid_records = silver_audit["invalid_records"],
    source_year_mismatches = (
        silver_audit["source_year_mismatches"]
    ),
    negative_dispatch_times = (
        silver_audit["negative_dispatch_times"]
    ),
    negative_incident_times = (
        silver_audit["negative_incident_times"]
    ),
    negative_travel_times = (
        silver_audit["negative_travel_times"]
    ),
    minimum_incident_datetime = (
        silver_audit["minimum_incident_datetime"]
    ),
    maximum_incident_datetime = (
        silver_audit["maximum_incident_datetime"]
    ),
    validation_status = audit_status
)

df_silver_audit = spark.createDataFrame([audit_record])

display(df_silver_audit)

StatementMeta(, 1c1a96c0-948f-4f2f-879d-8aeb5e83b856, 27, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 75a91dda-5dd6-405f-9d51-fbfc8b86c198)

In [26]:
# Write into audit table
(
    df_silver_audit
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(silver_audit_table_name)
)

print(
    f"Successfully updated audit table: "
    f"{silver_audit_table_name}"
)

StatementMeta(, 1c1a96c0-948f-4f2f-879d-8aeb5e83b856, 28, Finished, Available, Finished, False)

Successfully updated audit table: silver_ems_data_quality_audit


In [27]:
# Final Confirmation
print("=" * 60)
print("SILVER LAYER COMPLETION SUMMARY")
print("=" * 60)

print(f"Silver table: {silver_table_name}")
print(f"Audit table: {silver_audit_table_name}")
print(f"Total rows: {silver_audit['total_rows']:,}")
print(f"Total columns: {len(df_silver_saved.columns)}")
print(
    "Distinct incident IDs: "
    f"{silver_audit['distinct_incident_ids']:,}"
)
print(
    "Missing incident IDs: "
    f"{silver_audit['missing_incident_ids']:,}"
)
print(
    "Duplicate incident IDs: "
    f"{silver_audit['duplicate_incident_ids']:,}"
)
print(f"Validation status: {audit_status}")

print("=" * 60)

if audit_status != "PASS":
    raise ValueError(
        "Silver validation failed. "
        "Do not continue to the Gold layer."
    )

print("Silver layer completed successfully.")

StatementMeta(, 1c1a96c0-948f-4f2f-879d-8aeb5e83b856, 29, Finished, Available, Finished, False)

SILVER LAYER COMPLETION SUMMARY
Silver table: silver_ems_incidents
Audit table: silver_ems_data_quality_audit
Total rows: 10,881,496
Total columns: 63
Distinct incident IDs: 10,881,496
Missing incident IDs: 0
Duplicate incident IDs: 0
Validation status: PASS
Silver layer completed successfully.
